# Day 2 — Using AI APIs (OpenAI, Claude, and Hugging Face)

---

Yesterday: what an LLM is.
Today: **actually talk to one from Python.**

In this hour you'll learn:

1. How to safely store your API keys.
2. How to send a message to **ChatGPT (OpenAI)** and get a reply.
3. How to send the same message to **Claude (Anthropic)** and compare answers.
4. What **Hugging Face** is and why it matters.

By the end you can call the two biggest hosted AI providers with 5 lines of code each.

In [ ]:
!pip install openai anthropic python-dotenv --quiet

## 1. Store your keys in a `.env` file (never commit them!)

AI providers give you an API key that looks like this: `sk-abc123...`. 

**Rule:** never paste this into your code. Never commit it to GitHub. 

Create a file named `.env` in the folder where you're running Python:

```env
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
TOGETHER_API_KEY=...     # for Day 3 onward
```

Then add `.env` to your `.gitignore`. Your keys stay on your laptop.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file into environment variables

print("OpenAI key set?", bool(os.getenv("OPENAI_API_KEY")))
print("Claude key set?", bool(os.getenv("ANTHROPIC_API_KEY")))

## 2. Say hi to ChatGPT (OpenAI)

OpenAI's Python client is 3 lines: import, make a client, call `.create()`.

Every request has:
- **`model`** — which brain to use (`gpt-4o-mini` is the cheap default).
- **`messages`** — a conversation as a list. Each message has a `role` (`system`, `user`, or `assistant`) and `content`.

**System message** = the LLM's job description.
**User message** = what the human said.
**Assistant message** = what the AI said (used to continue a chat).

In [ ]:
from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY from env

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a friendly assistant. Reply in under 30 words."},
        {"role": "user",   "content": "Explain a black hole to a 10-year-old."},
    ],
)

print(resp.choices[0].message.content)
print("\nTokens used:", resp.usage.total_tokens)

## 3. Say hi to Claude (Anthropic)

Almost the same idea, one small difference: the system message is a **top-level parameter**, not inside `messages`.

| | OpenAI | Claude |
|---|---|---|
| System prompt | inside `messages` | separate `system=...` param |
| Text you get back | `resp.choices[0].message.content` | `resp.content[0].text` |
| Model name | `gpt-4o-mini` | `claude-3-5-haiku-latest` |

In [ ]:
from anthropic import Anthropic

aclient = Anthropic()  # picks up ANTHROPIC_API_KEY from env

resp = aclient.messages.create(
    model="claude-3-5-haiku-latest",
    max_tokens=200,
    system="You are a friendly assistant. Reply in under 30 words.",
    messages=[{"role": "user", "content": "Explain a black hole to a 10-year-old."}],
)

print(resp.content[0].text)
print("\nInput tokens :", resp.usage.input_tokens)
print("Output tokens:", resp.usage.output_tokens)

## 4. Which one should you use?

There is no single best model. There's the best model *for the task*.

| Task | Good first pick | Why |
|---|---|---|
| Short chat, high volume | `gpt-4o-mini` | Cheapest fluent model |
| Careful reasoning / code | `claude-3-5-sonnet-latest` | Best at multi-step thinking |
| Very long document | `claude-3-5-sonnet-latest` | Handles 200k tokens comfortably |
| Creative writing | `gpt-4o` | Slightly more "personality" |
| Sensitive data (stay local) | Open source models (Day 3) | Free + private |

**Real-world example:** Cursor (the AI code editor) routes coding requests to Claude and chit-chat to a smaller model. That's the exact skill you're learning.

## 5. Multi-turn conversations

LLMs don't remember anything between calls. If you want a *conversation*, you send the whole history every time.

That means: append the assistant's reply to your `messages` list, then send it back with the next user message.

In [ ]:
history = [
    {"role": "system", "content": "You are a helpful travel guide."},
]

def ask(user_msg):
    history.append({"role": "user", "content": user_msg})
    r = client.chat.completions.create(model="gpt-4o-mini", messages=history)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print("Q1:", ask("I want to visit Japan in April. Best city?"))
print("\nQ2:", ask("How many days should I spend there?"))
print("\nQ3:", ask("What was the first city you recommended?"))  # tests memory

**Notice:** the LLM "remembers" the answer to Q1 in Q3 — but only because we're sending the whole history each time. You are paying tokens for that memory.

## 6. What is Hugging Face?

**Hugging Face** is the "GitHub for AI models" — a website where people share thousands of open-source models you can download and use.

You'll typically use it in two ways:

1. **Browse the catalogue.** Pick a model like `sentence-transformers/all-MiniLM-L6-v2` (small, free, runs on your laptop).
2. **Download and use with the `transformers` library.** Or, easier: use a service that hosts them for you (that's **Together AI**, which we cover tomorrow).

**Why it matters:**
- Free — no per-token cost.
- Private — data never leaves the model.
- Customizable — you can fine-tune them.

We won't run models locally today (they're big and slow on a laptop). Tomorrow we'll use **Together AI** to talk to open-source models like LLaMA 3 through a simple API — best of both worlds.

## Recap

- Store API keys in `.env`, load with `python-dotenv`.
- **OpenAI**: `client.chat.completions.create(model, messages)`.
- **Claude**: `client.messages.create(model, system, messages, max_tokens)`.
- LLMs are stateless — you send the full conversation each time.
- Different models for different jobs — pick the cheapest one that works.
- **Hugging Face** hosts open-source models; **Together AI** (tomorrow) makes them easy to call.

Next: Together AI + prompt engineering — the skill that separates "AI hobbyist" from "AI engineer".